# Taller: Anatomía de un intérprete
## Análisis del código fuente de Picol

**Curso:** IS753 – Compiladores  
**Duración estimada:** 3 horas  
**Modalidad:** Individual o en parejas  
**Archivo requerido:** `picol.c`

---

## Propósito

En este taller se estudiará **Picol**, una implementación pequeña de un intérprete similar a Tcl escrita en C.

El objetivo es localizar en un programa real los componentes que normalmente aparecen en un compilador o intérprete:

1. Entrada del programa fuente.
2. Análisis léxico.
3. Análisis sintáctico.
4. Representación interna.
5. Gestión de variables y ámbitos.
6. Evaluación.
7. Ejecución de comandos.
8. Manejo de errores.
9. Gestión dinámica de memoria.

> Guarde `picol.c` en la misma carpeta de este cuaderno antes de ejecutar las celdas.

## Resultados de aprendizaje

Al finalizar el taller, el estudiante podrá:

- Diferenciar un compilador de un intérprete.
- Identificar el lexer y el parser dentro de un programa escrito en C.
- Explicar cómo Picol reconoce comandos, palabras, variables y bloques.
- Describir cómo se almacenan variables, procedimientos y comandos.
- Reconstruir el flujo de ejecución de un programa Tcl.
- Comparar la arquitectura de Picol con la de un compilador tradicional.

## 1. Preparación del entorno

In [1]:
from pathlib import Path
import re

ARCHIVO = Path("picol.c")

if not ARCHIVO.exists():
    raise FileNotFoundError(
        "No se encontró picol.c. Copie el archivo en la misma carpeta "
        "del cuaderno y vuelva a ejecutar esta celda."
    )

codigo = ARCHIVO.read_text(encoding="utf-8", errors="replace")
lineas = codigo.splitlines()

print(f"Archivo cargado: {ARCHIVO}")
print(f"Número de líneas: {len(lineas)}")
print(f"Número de caracteres: {len(codigo)}")

Archivo cargado: picol.c
Número de líneas: 809
Número de caracteres: 26727


### Pregunta 1

Antes de analizar el código, responda:

1. ¿Picol es principalmente un compilador, un intérprete o una máquina virtual?
2. ¿Qué diferencia existe entre compilar un programa e interpretarlo?
3. ¿Qué salida espera obtener Picol después de procesar un script?

**Respuesta:**

1. Picol es un interprete
2. Compilar un programa es traducir el código fuente y a partir de este generar un ejecutable, lo que quiere decir que ya no necesito el compilador, por otro lado el interprete: lee, traduce y ejecuta todas las instrucciones, por lo cual es dependiente del interprete
3. Al ser un interprete debera mostrar en pantalla lo que el codigo fuente pretendia mostrar 


## 2. Exploración general del archivo

In [2]:
for numero, linea in enumerate(lineas[:80], start=1):
    print(f"{numero:4}: {linea}")

   1: /* Tcl in ~ 500 lines of code.
   2:  *
   3:  * IMPORTANT: this is Picol version 2! For the original code, check
   4:  * the commit history of this repository.
   5:  *
   6:  * Copyright (c) 2007-2026, Salvatore Sanfilippo <antirez at gmail dot com>
   7:  * All rights reserved.
   8:  *
   9:  * Redistribution and use in source and binary forms, with or without
  10:  * modification, are permitted provided that the following conditions are met:
  11:  *
  12:  *   * Redistributions of source code must retain the above copyright notice,
  13:  *     this list of conditions and the following disclaimer.
  14:  *   * Redistributions in binary form must reproduce the above copyright
  15:  *     notice, this list of conditions and the following disclaimer in the
  16:  *     documentation and/or other materials provided with the distribution.
  17:  *
  18:  * THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS AND CONTRIBUTORS "AS IS"
  19:  * AND ANY EXPRESS OR IMPLIED WARRANTIE

In [3]:
patron_funcion = re.compile(
    r"^\s*(?:static\s+)?(?:int|void|char\s*\*|const\s+char\s*\*|"
    r"struct\s+\w+\s*\*|\w+\s*\*)\s+"
    r"([A-Za-z_]\w*)\s*\([^;]*\)\s*\{",
    re.MULTILINE
)

funciones = patron_funcion.findall(codigo)

print(f"Funciones encontradas aproximadamente: {len(funciones)}")
for nombre in funciones:
    print("-", nombre)

Funciones encontradas aproximadamente: 28
- picolInitParser
- picolParseSep
- picolParseEol
- picolParseCommand
- picolParseVar
- picolParseBrace
- picolParseString
- picolParseComment
- picolGetToken
- picolSetResult
- picolSetVar
- picolRegisterCommand
- picolEval
- picolExprExpansion
- picolArityErr
- picolCommandExpr
- picolCommandSet
- picolCommandPuts
- picolCommandIf
- picolCommandWhile
- picolCommandRetCodes
- picolDropCallFrame
- picolFreeInterp
- picolCommandCallProc
- picolCommandProc
- picolCommandReturn
- picolRegisterCoreCommands
- main


### Actividad 2

Observe la lista de funciones y clasifíquelas inicialmente.

| Categoría | Funciones candidatas |
|---|---|
| Lexer o tokenización |picolGetToken |
| Parser |  picolInitParser, picolParseSep, picolParseEol, picolParseCommand, picolParseVar, picolParseBrace, picolParseString, picolParseComment|
| Evaluación | picolEval|
| Variables | picolSetVar|
| Procedimientos | picolCommandCallProc, picolCommandProc|
| Manejo de errores | picolCommandRetCodes, picolArityErr|
| Memoria | picolRegisterCommand|

> Esta primera clasificación es provisional. Será revisada al final del taller.

## 3. Herramientas para inspeccionar el código

In [5]:
def buscar(texto, contexto=3, ignorar_mayusculas=True):
    # Busca un texto o una expresión regular y muestra las líneas cercanas.
    flags = re.IGNORECASE if ignorar_mayusculas else 0
    patron = re.compile(texto, flags)

    encontrados = 0
    for i, linea in enumerate(lineas):
        if patron.search(linea):
            encontrados += 1
            inicio = max(0, i - contexto)
            fin = min(len(lineas), i + contexto + 1)

            print("=" * 78)
            for j in range(inicio, fin):
                marca = ">>" if j == i else "  "
                print(f"{marca} {j + 1:4}: {lineas[j]}")

    if encontrados == 0:
        print("No se encontraron coincidencias.")
    else:
        print(f"\nCoincidencias: {encontrados}")


def mostrar_rango(inicio, fin):
    # Muestra un rango de líneas. Los números se interpretan desde 1.
    inicio = max(1, inicio)
    fin = min(len(lineas), fin)

    for numero in range(inicio, fin + 1):
        print(f"{numero:4}: {lineas[numero - 1]}")

In [6]:
buscar(r"picolEval", contexto=4)

    378:     }
    379: }
    380: 
    381: /* EVAL! */
>>  382: int picolEval(struct picolInterp *i, char *t) {
    383:     struct picolParser p;
    384:     int argc = 0, j;
    385:     char **argv = NULL;
    386:     char errbuf[1024];
    414:             }
    415:             free(t);
    416:             t = xstrdup(v->val);
    417:         } else if (p.type == PT_CMD) {
>>  418:             retcode = picolEval(i,t);
    419:             free(t);
    420:             if (retcode != PICOL_OK) goto err;
    421:             t = xstrdup(i->result);
    422:         } else if (p.type == PT_ESC) {
    553:     i->level--;
    554:     return a;
    555: }
    556: 
>>  557: /* Trick: wrap 's' as "expr <s>" and evaluate it, so that picolEval handles
    558:  * $var and [cmd] substitution before expr parses pure math expression.
    559:  * This is used in [if] and [while] condition evaluation. */
    560: int picolExprExpansion(struct picolInterp *i, char *s) {
    561:     int

## 4. Identificación del lexer

Un **lexer** o analizador léxico recibe caracteres y reconoce unidades significativas.

En Picol estas unidades pueden incluir:

- Palabras.
- Separadores.
- Variables.
- Cadenas entre comillas.
- Bloques entre llaves.
- Sustitución de comandos.
- Finales de comando.

In [7]:
terminos_lexer = [
    r"parse",
    r"token",
    r"separator",
    r"brace",
    r"quote",
    r"variable",
    r"command",
    r"eol",
]

for termino in terminos_lexer:
    print("\n" + "#" * 78)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=1)


##############################################################################
BÚSQUEDA: parse
     73: 
>>   74: struct picolParser {
     75:     char *text;         // The program to parse
     74: struct picolParser {
>>   75:     char *text;         // The program to parse
     76:     char *p;            // Current parsing position in 'text'
    113: 
>>  114: void picolInitParser(struct picolParser *p, char *text) {
    115:     p->text = p->p = text;
    120: 
>>  121: int picolParseSep(struct picolParser *p) {
    122:     p->start = p->p;
    130: 
>>  131: int picolParseEol(struct picolParser *p) {
    132:     p->start = p->p;
    142: 
>>  143: int picolParseCommand(struct picolParser *p) {
    144:     int level = 1;
    172: 
>>  173: int picolParseVar(struct picolParser *p) {
    174:     p->start = ++p->p; p->len--; /* skip the $ */
    192: 
>>  193: int picolParseBrace(struct picolParser *p) {
    194:     int level = 1;
    215: 
>>  216: int picolParseString(struc

### Actividad 4

Complete la tabla con los nombres reales encontrados en `picol.c`.

| Parte léxica | Función o constante | Explicación |
|---|---|---|
| Inicio del parser | picolInitParser| recibe el texto que el parser evaluara|
| Separadores | picolParseSep| separa argumentos|
| Palabras simples | picolParseString| evalua tokens que pueden ser de tipo PT_SEP, PT_EOL o PT_str toma acciones si es '{' o '"'|
| Variables | picolParseVar| se salta el primer y caracter de una string|
| Comillas | picolparseQuote| Evalua si hay algo entre comillas|
| Llaves | picolparseQuote| Evalua si hay algo entre llaves|
| Sustitución de comandos | picolRegisterCommand| registra un comando, salta al siguiente si no existe, si no hay argumentos pasados por consola pone el resultado como un error de buffer|
| Fin de línea o comando | EOL| indica cuando hay un salto de linea o termina un comando|

Responda:

1. ¿Picol crea objetos `Token` independientes?
creo que todo hace parte de una estructura general, 
2. ¿El lexer genera primero una lista completa de tokens?
se supone deberia hacerlos primero
3. ¿Lexer y parser están completamente separados?
el parser depende del lexer, el uno sin el otro no son nada, uno hace tokenizacion y el otro analiza si estos estan organizados
4. ¿Qué información guarda el estado del parser?
recuerda que ya ha procesado

## 5. Identificación del parser

In [8]:
buscar(r"picolGetToken", contexto=12)

    257:         p->p++; p->len--;
    258:     }
    259:     return PICOL_OK; /* unreached */
    260: }
    261: 
    262: int picolParseComment(struct picolParser *p) {
    263:     while(p->len && *p->p != '\n') {
    264:         p->p++; p->len--;
    265:     }
    266:     return PICOL_OK;
    267: }
    268: 
>>  269: int picolGetToken(struct picolParser *p) {
    270:     while(1) {
    271:         if (!p->len) {
    272:             if (p->type != PT_EOL && p->type != PT_EOF)
    273:                 p->type = PT_EOL;
    274:             else
    275:                 p->type = PT_EOF;
    276:             return PICOL_OK;
    277:         }
    278:         switch(*p->p) {
    279:         case ' ': case '\t':
    280:             if (p->insidequote) return picolParseString(p);
    281:             return picolParseSep(p);
    387:     int retcode = PICOL_OK;
    388:     picolSetResult(i,"");
    389:     if (++i->level > PICOL_MAX_RECURSION_LEVEL) {
    390:         i->l

In [9]:
buscar(r"picolParse", contexto=8)

     66:     PT_STR, // String without escapes, no post processing needed.
     67:     PT_CMD, // Command, that is [.... something ...]
     68:     PT_VAR, // Variable like $var
     69:     PT_SEP, // Arguments separator
     70:     PT_EOL, // End of command
     71:     PT_EOF  // End of file (stops the parsing loop)
     72: };
     73: 
>>   74: struct picolParser {
     75:     char *text;         // The program to parse
     76:     char *p;            // Current parsing position in 'text'
     77:     int len;            // Remaining length
     78:     char *start;        // Token start
     79:     char *end;          // Token end
     80:     int type;           // Token type, PT_...
     81:     int insidequote;    // True if inside " "
     82: };
    106: 
    107: struct picolInterp {
    108:     int level; /* Level of nesting */
    109:     struct picolCallFrame *callframe;
    110:     struct picolCmd *commands;
    111:     char *result;
    112: };
    113: 
>>  

### Actividad 5

Explique el flujo del parser:

```text
Texto fuente
    ↓
________________________
    ↓
________________________
    ↓
Token o fragmento reconocido
    ↓
________________________
```

¿Picol usa una gramática BNF explícita como Yacc, Bison o SLY?  
Explique la diferencia entre un parser generado y el parser manual de Picol.

## 6. ¿Existe un árbol de sintaxis abstracta?

In [ ]:
for termino in [r"AST", r"Node", r"Statement", r"Expression", r"Program"]:
    print("\n" + "-" * 60)
    print(f"Búsqueda de: {termino}")
    buscar(termino, contexto=1)

### Actividad 6

Responda:

1. ¿Existe una estructura que represente un AST completo?
2. ¿Picol analiza primero todo el programa y lo ejecuta después?
3. ¿Qué ventaja obtiene al no construir un AST?
4. ¿Qué limitaciones produce esta decisión?
5. ¿La lista de palabras de un comando puede considerarse una representación intermedia mínima?

## 7. El ciclo de evaluación

In [ ]:
buscar(r"int\s+picolEval|picolEval\s*\(", contexto=25)

### Actividad 7

Describa paso a paso la función de evaluación.

Después explique qué ocurre al ejecutar:

```tcl
set x 10
puts $x
```

Incluya:

- Lectura del comando `set`.
- Construcción de argumentos.
- Almacenamiento de `x`.
- Lectura de `puts`.
- Sustitución de `$x`.
- Búsqueda y ejecución de `puts`.

## 8. Tabla de comandos

In [ ]:
buscar(r"picolRegister", contexto=12)
buscar(r"struct\s+picolCmd|picolCmd", contexto=8)

### Actividad 8

| Elemento | Nombre en Picol | Función |
|---|---|---|
| Estructura de comando | | |
| Registro de comando | | |
| Búsqueda de comando | | |
| Función C asociada | | |
| Datos privados | | |

Responda cómo se asocia el nombre de un comando con una función de C y qué sucede cuando el comando no existe.

## 9. Variables y ámbitos

In [ ]:
for termino in [r"picolVar", r"picolGetVar", r"picolSetVar", r"callframe", r"CallFrame"]:
    print("\n" + "#" * 70)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=6)

### Actividad 9

Complete el diagrama:

```text
Intérprete
    ↓
Marco de llamada actual
    ↓
Lista de variables
    ↓
Nombre ───────── Valor
```

Responda:

1. ¿Cómo se representa una variable?
2. ¿Se usa un arreglo, tabla hash o lista enlazada?
3. ¿Cómo se implementan los ámbitos locales?
4. ¿Qué ocurre cuando se llama un procedimiento?

## 10. Procedimientos definidos por el usuario

In [ ]:
for termino in [r"picolProc", r"proc", r"picolCallProc"]:
    print("\n" + "#" * 70)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=10)

### Actividad 10

Analice:

```tcl
proc cuadrado {x} {
    expr {$x * $x}
}
```

| Elemento del procedimiento | Representación en Picol |
|---|---|
| Nombre | |
| Lista de parámetros | |
| Cuerpo | |
| Datos privados del comando | |
| Marco de llamada | |

Explique cuándo se interpreta el cuerpo y cómo se asignan argumentos a parámetros.

## 11. Comandos incorporados

In [ ]:
for termino in [
    r"picolCommandSet",
    r"picolCommandPuts",
    r"picolCommandIf",
    r"picolCommandWhile",
    r"picolCommandProc",
    r"picolCommandReturn",
]:
    print("\n" + "=" * 70)
    print(termino)
    buscar(termino, contexto=8)

### Actividad 11

| Comando | Función C | Validación de argumentos | Operación |
|---|---|---|---|
| `set` | | | |
| `if` | | | |
| `while` | | | |

¿`if` y `while` forman parte del parser o se implementan como comandos normales? Explique.

## 12. Expresiones

In [ ]:
buscar(r"expr", contexto=10)

### Actividad 12

1. ¿Existe un parser independiente para expresiones?
2. ¿Qué operadores soporta?
3. ¿Existe precedencia?
4. ¿Cómo se procesan los operandos?
5. ¿La expresión se compila o se evalúa directamente?

## 13. Manejo de errores y códigos de retorno

In [ ]:
for termino in [
    r"PICOL_OK",
    r"PICOL_ERR",
    r"PICOL_RETURN",
    r"PICOL_BREAK",
    r"PICOL_CONTINUE",
    r"wrong # args",
    r"unknown command",
]:
    print("\n" + "#" * 70)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=4)

### Actividad 13

| Código | Significado | Ejemplo |
|---|---|---|
| `PICOL_OK` | | |
| `PICOL_ERR` | | |
| `PICOL_RETURN` | | |
| `PICOL_BREAK` | | |
| `PICOL_CONTINUE` | | |

Explique cómo se almacena y propaga un error.

## 14. Gestión de memoria

In [ ]:
for termino in [r"malloc", r"calloc", r"realloc", r"free", r"strdup"]:
    print("\n" + "#" * 70)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=2)

### Actividad 14

| Estructura o dato | Reserva | Liberación | Riesgo |
|---|---|---|---|
| | | | |
| | | | |
| | | | |
| | | | |
| | | | |

Identifique los datos que permanecen durante toda la vida del intérprete y los que solo existen durante una llamada.

## 15. Función principal e inicialización

In [ ]:
buscar(r"main\s*\(", contexto=25)

for termino in [r"picolInitInterp", r"picolRegister", r"argc", r"argv"]:
    print("\n" + "#" * 70)
    print(f"BÚSQUEDA: {termino}")
    buscar(termino, contexto=8)

### Actividad 15

Complete:

```text
main
  ↓
Creación del intérprete
  ↓
________________________________
  ↓
Registro de comandos
  ↓
Lectura del script
  ↓
________________________________
  ↓
Presentación del resultado
```

Explique cómo se usan `argc` y `argv`.

## 16. Correspondencia con las fases de un compilador

| Fase tradicional | ¿Existe? | Función, estructura o mecanismo | Observaciones |
|---|---:|---|---|
| Lectura del programa | | | |
| Análisis léxico | | | |
| Análisis sintáctico | | | |
| AST | | | |
| Tabla de símbolos | | | |
| Análisis semántico | | | |
| Representación intermedia | | | |
| Optimización | | | |
| Generación de código | | | |
| Máquina virtual | | | |
| Ejecución directa | | | |
| Manejo de errores | | | |

### Pregunta central

¿Por qué Picol puede ejecutar programas sin tener todas las fases de un compilador tradicional?

**Extensión mínima:** 200 palabras.

## 17. Diagrama de arquitectura

Complete con nombres reales:

```text
                         Script Tcl
                             │
                             ▼
                   Estado del parser
                   __________________
                             │
                             ▼
                 Reconocimiento de palabras
                 __________________________
                             │
                             ▼
               Sustitución de variables/comandos
               ________________________________
                             │
                             ▼
                   Lista de argumentos
                   __________________
                             │
                             ▼
                   Búsqueda del comando
                   __________________
                             │
                             ▼
                  Función C o procedimiento
                  _________________________
                             │
                             ▼
                     Resultado o error
```

## 18. Prueba experimental

Compile desde una terminal:

```bash
gcc -Wall -Wextra -O0 -g picol.c -o picol
```

Registre:

- Comando de compilación.
- Advertencias.
- Comando de ejecución.
- Salida.
- Errores encontrados.

In [ ]:
print("gcc -Wall -Wextra -O0 -g picol.c -o picol")

## 19. Extensión opcional: comando `square`

Implemente:

```tcl
square 10
```

Resultado:

```text
100
```

Debe entregar:

1. Función C.
2. Registro del comando.
3. Prueba.
4. Explicación.
5. Manejo de argumentos inválidos.

## 20. Preguntas de reflexión

1. ¿Qué parte corresponde al lexer?
2. ¿Qué parte corresponde al parser?
3. ¿Por qué están tan unidos?
4. ¿Qué estructura actúa como tabla de símbolos?
5. ¿Cómo se representan los procedimientos?
6. ¿Por qué `if` y `while` pueden ser comandos?
7. ¿Qué se gana al interpretar directamente?
8. ¿Qué se pierde al no construir un AST?
9. ¿Qué cambios permitirían convertir Picol en compilador?
10. ¿Cuál fue la parte más difícil de identificar?

## 21. Entregables y rúbrica

### Entregables

1. Cuaderno ejecutado y respondido.
2. Archivo `picol.c`.
3. Diagrama de arquitectura.
4. Tabla de fases.
5. Flujo de `set x 10; puts $x`.
6. Extensión `square`, cuando sea solicitada.

### Rúbrica

| Criterio | Porcentaje |
|---|---:|
| Identificación del lexer | 20 % |
| Identificación del parser | 20 % |
| Ciclo de evaluación | 20 % |
| Variables, ámbitos y procedimientos | 15 % |
| Comparación con un compilador | 15 % |
| Claridad y presentación | 10 % |
| **Total** | **100 %** |

## Conclusión del estudiante

Redacte una conclusión general sobre la arquitectura de Picol y su utilidad para aprender compiladores e intérpretes.

**Extensión mínima:** 250 palabras.